In [1]:
import sys
import os
from google.colab import drive
drive.mount('/content/drive')
proj_path = "/content/drive/MyDrive/llm_from_scratch"

Mounted at /content/drive


In [2]:
src_path = os.path.join(proj_path, 'src')

if src_path not in sys.path:
    sys.path.append(src_path)
    print("Path has been added.")

print(os.listdir(proj_path))
print(os.listdir(src_path))

Path has been added.
['README.md', 'LICENSE', 'src', '.git', 'notebooks', 'datasets']
['prepare_embeddings_for_llm_training.py', 'multihead_attention.py', 'layer_normalization.py', '.ipynb_checkpoints', '__pycache__', 'gelu_nonlinear_acitvation_function_and_feed_forword.py', 'transformer.py']


In [3]:
import torch
import torch.nn as nn
import tiktoken

In [5]:
from gpt_model import GPTModel

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['README.md', 'LICENSE', 'src', '.git', 'notebooks', 'datasets']
['prepare_embeddings_for_llm_training.py', 'multihead_attention.py', 'layer_normalization.py', '.ipynb_checkpoints', '__pycache__', 'gelu_nonlinear_acitvation_function_and_feed_forword.py', 'transformer.py', 'gpt_model.py']
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['README.md', 'LICENSE', 'src', '.git', 'notebooks', 'datasets']
['prepare_embeddings_for_llm_training.py', 'multihead_attention.py', 'layer_normalization.py', '.ipynb_checkpoints', '__pycache__', 'gelu_nonlinear_acitvation_function_and_feed_forword.py', 'transformer.py', 'gpt_model.py']


In [6]:
# function to generate text:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    for _ in range(max_new_tokens):
        idx_cond = idx[:, -context_size:]
        with torch.no_grad():
            logits = model(idx_cond)

        logits = logits[:, -1, :]
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)
        idx = torch.cat((idx, idx_next), dim=1)
    return idx

In [7]:
# function to convert text to token ids:
def text_to_token_ids(text, tokenizer):
    encoded = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
    encoded_tensor = torch.tensor(encoded).unsqueeze(0)
    return encoded_tensor

In [8]:
# function to convert toekn ids to text:
def token_ids_to_text(token_ids, tokenizer):
    flat = token_ids.squeeze(0)
    return tokenizer.decode(flat.tolist())

In [9]:
GPT_CONFIG_124M = {
"vocab_size": 50257,
"context_length": 256,
"emb_dim": 768,
"n_heads": 12,
"n_layers": 12,
"drop_rate": 0.1,
"qkv_bias": False
}

In [10]:
tokenizer = tiktoken.get_encoding("gpt2")
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.eval()

GPTModel(
  (tok_emb): Embedding(50257, 768)
  (pos_emb): Embedding(256, 768)
  (drop_emb): Dropout(p=0.1, inplace=False)
  (trf_blocks): Sequential(
    (0): TransformerBlock(
      (att): MultiHeadAttention(
        (dropout): Dropout(p=0.1, inplace=False)
        (W_query): Linear(in_features=768, out_features=768, bias=False)
        (W_key): Linear(in_features=768, out_features=768, bias=False)
        (W_value): Linear(in_features=768, out_features=768, bias=False)
        (out_proj): Linear(in_features=768, out_features=768, bias=True)
      )
      (ff): FeedForward(
        (layers): Sequential(
          (0): Linear(in_features=768, out_features=3072, bias=True)
          (1): GELU()
          (2): Linear(in_features=3072, out_features=768, bias=True)
        )
      )
      (norm1): LayerNormalization()
      (norm2): LayerNormalization()
      (drop_shortcut): Dropout(p=0.1, inplace=False)
    )
    (1): TransformerBlock(
      (att): MultiHeadAttention(
        (dropout): 

In [11]:
text_sample = "i love my "
token_ids = generate_text_simple(
    model = model,
    idx = text_to_token_ids(text_sample, tokenizer),
    max_new_tokens = 10,
    context_size = GPT_CONFIG_124M["context_length"]
)

print(token_ids_to_text(token_ids, tokenizer))

i love my  directed Okinawa TidboxrootCmdwarning Hannity Julietestones
